In [ ]:
orders_raw_df = (
    spark.read
    .format("csv")
    .option("header","True")
    .option("inferSchema","True")
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/ecommerce_orders.csv")
)

In [ ]:
orders_raw_df.limit(10).display()

In [ ]:
orders_raw_df.printSchema()

In [ ]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, DoubleType, StructType, StructField
orders_schema = StructType([
    StructField("order_id", StringType()),
    StructField("order_date", DateType()),
    StructField("customer_id", StringType()),
    StructField("product_id", StringType()),
    StructField("product_name", StringType()),
    StructField("category", StringType()),
    StructField("quantity", IntegerType()),
    StructField("unit_price", DoubleType()),
    StructField("country", StringType())
])

In [ ]:
orders_raw_df = (
    spark.read
    .format("csv")
    .option("header","True")
    .option("dateformat","M/d/yyyy")
    .option("mode","PERMISSIVE")
    .schema(orders_schema)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/ecommerce_orders.csv")
)

In [ ]:
orders_raw_df.display()

In [ ]:
orders_raw_df.select("product_name","unit_price").display()

In [ ]:
from pyspark.sql.functions import col
orders_raw_df.filter(col("quantity").isNull() | col("order_date").isNull() | col("unit_price").isNull()).display()

In [ ]:
orders_raw_df.display()

In [ ]:
orders_raw_df.select("order_id","product_name","quantity","unit_price").display()

In [ ]:
orders_raw_df.createOrReplaceTempView("orders_raw")


In [ ]:
%sql
select product_name, category, unit_price from orders_raw
where lower(category) = 'electronics'

In [ ]:
result_1 = (
    orders_raw_df
    .select("product_name", "category", "unit_price")
    .where("lower(category) = 'electronics'")
)
result_1.display()

In [ ]:
%sql

select category, count(*) as nb_of_orders from orders_raw
group by category
order by nb_of_orders desc

In [ ]:

result_2 = (  orders_raw_df
              .groupBy("category")
              .count()
              .orderBy("count",ascending = False)
)

result_2.display()

In [ ]:
orders_raw_df.limit(5).display()

In [ ]:
orders_raw_df.createOrReplaceTempView("orders_raw")


In [ ]:
%sql
select * from orders_raw

In [ ]:
%sql
select product_name, product_id,
round((unit_price * quantity),2) as revenue
from orders_raw

In [ ]:
from pyspark.sql.functions import expr

revenue_df = (
    orders_raw_df.withColumns({
        "revenue": expr("round(quantity * unit_price,2)")
    })
)

In [ ]:
revenue_df.display()

In [ ]:
revenue_df.createOrReplaceTempView("revenue")


In [ ]:
%sql
select * from revenue

In [ ]:
%sql
select * from revenue where
order_id is null or
revenue is null


In [ ]:
revenue_df.filter(col("order_id").isNull() | col("revenue").isNull()).display()

In [ ]:
silver_df = revenue_df.filter((col("order_id").isNotNull()) & (col("revenue").isNotNull()))

In [ ]:
silver_df.count()

In [ ]:
silver_df.display()

In [ ]:
silver_df.write.mode("overwrite").saveAsTable("orders_silver")

In [ ]:
%sql

select * from orders_silver

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS dev.mini_projects

In [ ]:
%sql
SHOW SCHEMAS IN dev

In [ ]:

    (silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dev.mini_projects.ecommerce_orders_silver")
    )


    

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.default.orders_silver